This notebook first allows an overview of the open-ephys recording's analog data channels, and synchronizes arena events to the experimental timebase via the photodiode analog channel. This channel holds the output of a photodiode connected to the screen and will allow full sync verification against all other data sources, which will also be dealt with in this pipeline

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
# All imports consolidated here for the block synchronization pipeline

from __future__ import annotations
from pathlib import Path
from typing import Optional, Tuple, Literal, Sequence, Union
from dataclasses import dataclass
import re
import pickle

# Core data science
import numpy as np
import pandas as pd

# Bokeh for interactive visualization
from bokeh.io import output_notebook, show, reset_output
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, CustomJS, Span, HoverTool
from bokeh.palettes import Category10
try:
    # Bokeh 3.x
    from bokeh.models import Slider
    from bokeh.resources import INLINE
except Exception:
    # Bokeh 2.x
    from bokeh.models.widgets import Slider
    from bokeh.resources import INLINE
from bokeh.layouts import column, row

# Project utilities
from eye_tracking_system_tools.preprocessing import utility_functions as uf

# Optional visualization (for jitter analysis plots)
import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================================
# Block synchronization pipeline — all logic lives in block_sync_core and block_sync_visualization
# ============================================================================

from eye_tracking_system_tools.preprocessing.block_sync_core import (
    simple_sync_build,
    shift_eye_df_by_index,
    build_final_sync_df_merge_nearest,
    verify_final_df_against_sources,
    export_final_sync_df,
    load_final_sync_df,
    build_arena_grid_df,
    ArenaGridInfo,
    create_distance_plot,
    add_intermediate_elements,
    find_jittery_frames,
    export_eye_data_2d,
    describe_eye_tick,
)

from eye_tracking_system_tools.preprocessing.block_sync_visualization import (
    plot_simple_sync_bokeh,
    hover_inspect_eyes_bokeh,
    sanity_plot_final_df,
    insert_dup_by_pos,
    insert_dup_by_oe_sample,
    insert_duplicate_frames_slide,
    remove_frame_at_pos,
    interactive_sync_tool_bokeh,
    plot_sync_verification_with_electrophys,
    plot_led_off_events_viewer,
)


In [ ]:
# block instantiation:
bad_blocks = [] #
experiment_path = Path(r"D:\sample_data_for_eye_repo")

block_numbers = [19]
animal = 'PV_208'
block_collection = uf.block_generator(block_numbers=block_numbers,
                                      experiment_path=experiment_path,
                                      animal=animal,
                                      bad_blocks=bad_blocks
                                      )
for block in block_collection:
    block.channeldict = None
    if block.animal_call == 'PV_208':
        block.channeldict={1: 'LED_driver',
                           7: 'L_eye_TTL',
                           2: 'Arena_TTL',
                           8: 'R_eye_TTL'}
    elif block.animal_call == "TE_21":
        block.channeldict={1:'Arena_TTL',
                           4:'LED_driver',
                           5:'R_eye_TTL',
                           8:'L_eye_TTL'}
    elif block.animal_call == "PV_106":
        block.channeldict={1: 'LED_driver',
                           7: 'L_eye_TTL',
                           2: 'Arena_TTL',
                           8: 'R_eye_TTL'}
# create a block_dict object for ease of access:
block_dict = {}
for b in block_collection:
    block_dict[str(b.block_num)] = b

In [ ]:
def load_final_sync_df(block, filename=None, verbose=True):
    """
    Load a downstream-compatible final sync dataframe from disk and set:
      - block.final_sync_df
      - block.blocksync_df  (legacy compatibility)

    If `filename` is None, tries 'final_sync_df.csv' then 'blocksync_df.csv'
    inside `block.analysis_path`.

    Returns
    -------
    pd.DataFrame
    """
    # 1) pick a file
    ap = Path(block.analysis_path)
    candidates = [filename] if filename else ["final_sync_df.csv", "blocksync_df.csv"]
    path = None
    for name in candidates:
        p = ap / name
        if p.exists():
            path = p
            break
    if path is None:
        raise FileNotFoundError(f"No sync file found. Tried: {', '.join(str(ap / n) for n in candidates)}")

    # 2) read & validate schema
    df = pd.read_csv(path)
    required = ['Arena_TTL','Arena_frame','L_eye_frame','R_eye_frame','L_values','R_values']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    # 3) light coercions to keep downstream happy
    df = df.copy()
    df['Arena_TTL'] = df['Arena_TTL'].astype(float)

    # 4) set attributes
    block.final_sync_df = df
    block.blocksync_df = df  # some older code reads this
    if verbose:
        print(f"[OK] Loaded {path.name} → block.final_sync_df (rows={len(df):,})")

    return df
for block in block_collection:
    load_final_sync_df(block)

block.final_sync_df['ms_axis'] = block.final_sync_df['Arena_TTL'].values / (block.sample_rate / 1000)

In [ ]:
def load_eye_data(block):
    """
    Load the eye dataframes from CSV files created by the synchronization pipeline.
    No rotation matrices are loaded as rotation is no longer used.
    :param block: The current blocksync class
    :return: None
    """
    try:
        block.left_eye_data = pd.read_csv(block.analysis_path / 'left_eye_data_degrees_raw_verified.csv', index_col=0, engine='python')
        block.right_eye_data = pd.read_csv(block.analysis_path / 'right_eye_data_degrees_raw_verified.csv', index_col=0, engine='python')
        print(f'Loaded eye data for block {block.block_num}')
    except FileNotFoundError:
        print('Eye data files not found. Run the synchronization pipeline first!')
        raise

for block in block_collection:
    load_eye_data(block)

In [ ]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import Legend, ColumnDataSource
import numpy as np

output_notebook()

block = block_collection[0]
oe_rec = block.oe_rec

num_analog = len(oe_rec.analogChannelNumbers)
analog_channel_indices = oe_rec.analogChannelNumbers

# Get analog data for all channels for the entire session
# Try get_analog_data signature: get_analog_data(channels, start_samples, window_length)
# We'll fetch from sample 0 to oe_rec.recordLength for each channel

window_length = oe_rec.recordingDuration_ms


start_sample = [0]



analog_data, analog_timestamps = oe_rec.get_analog_data(analog_channel_indices, [start_sample], window_length)
total_samples = np.shape(analog_timestamps[0])[0]
channel_labels = [f"Analog {i}" for i in analog_channel_indices]

# 10x downsampling for faster plotting (get_analog_data still reads full data; this only reduces points drawn)
downsample_factor = 5
time_full = np.arange(total_samples) / block.sample_rate
time_ds = time_full[::downsample_factor]
data = {"time": time_ds}

# analog_data shape: (n_channels, n_windows, nSamples); we use window 0 and downsample each channel
for idx, ch in enumerate(analog_channel_indices):
    values = analog_data[idx, 0, :][::downsample_factor]  # (nSamples,) -> 1/10 points
    data[channel_labels[idx]] = values
source = ColumnDataSource(data)

colors = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
    "#bcbd22", "#17becf"
]

p = figure(
    width=900, height=450,
    title="Analog Channels (block 0)",
    x_axis_label="Time (s)",
    y_axis_label="Analog Value",
    tools="pan,wheel_zoom,box_zoom,reset,save,hover"
)

for i, label in enumerate(channel_labels):
    p.line('time', label, source=source, line_width=2, color=colors[i % len(colors)], legend_label=label)

p.legend.location = "top_right"
p.legend.click_policy = "hide"
show(p)

In [ ]:
data = oe_rec.get_analog_data(list(oe_rec.analogChannelNumbers),[[0]],int(oe_rec.recordingDuration_ms))

In [ ]:
analog_channel_indices[0]

In [ ]:
oe_rec.analogChannelNumbers

In [ ]:
# This produces an IndexError: "invalid index to scalar variable."
# According to the traceback (see context), the issue is that oe_rec.get_analog_data
# expects start_time_ms to have the same structure/dimensionality as the number of channels, 
# probably as a nested list/array: [[0]] instead of [0], or maybe oe_rec.analogChannelNumbers needs to be used.
# Let's check the available analog channels:
print("Channels present:", oe_rec.analogChannelNumbers)  # e.g., array([1,2,3,4,5,6,7,8])
channels = [1]
print("Is channel in file?:", all([c in oe_rec.channelNumbers for c in channels]))

# Try with the channels actually present
# Also, use nested lists for start_time_ms if get_analog_data expects this.
# If you get another error, check the function signature!
analog_data = oe_rec.get_analog_data(channels, [[0]], oe_rec.recordingDuration_ms)

In [ ]:
channels = [1]
all([c in oe_rec.channelNumbers for c in channels])

In [ ]:
analog_data